# AlphaTrain Pillar 2Q: Escape Velocity (v4+v5 Pure Self-Play)

Combined self-play: v4 (2N model, mean 2,625) + v5 (2P ep6, mean 5,117).
No expert data. 15 epochs. Checkpoint bracket at epochs 3, 6, 9, 12, 15.

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code
2. `selfplay_v4v5_combined.pt.gz` — combined self-play data
3. `pillar2p_ep6.pt` — base model (on Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

# Extract code (always fresh)
!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

# Copy model
os.makedirs('/content/alphatrain/data', exist_ok=True)
print('Copying model...')
shutil.copy(f'{DRIVE}/pillar2p_ep6.pt', '/content/alphatrain/data/model.pt')
assert os.path.exists('/content/alphatrain/data/model.pt'), 'Model copy failed!'
print(f'Model: {os.path.getsize("/content/alphatrain/data/model.pt")/1e6:.0f} MB')

# Decompress combined self-play data
print('Decompressing combined self-play data...')
t0 = time.time()
!gzip -dc {DRIVE}/selfplay_v4v5_combined.pt.gz > /content/alphatrain/data/selfplay.pt
print(f'Data: {os.path.getsize("/content/alphatrain/data/selfplay.pt")/1e9:.1f} GB ({time.time()-t0:.0f}s)')

# VERIFY
for f in ['model.pt', 'selfplay.pt']:
    path = f'/content/alphatrain/data/{f}'
    assert os.path.exists(path), f'MISSING: {path}'
    print(f'  OK: {f} ({os.path.getsize(path)/1e6:.0f} MB)')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

d = torch.load('/content/alphatrain/data/selfplay.pt', weights_only=True, map_location='cpu')
print(f'\nData: {d["boards"].shape[0]:,} states, '
      f'max_score={d["max_score"]}, bins={d["num_value_bins"]}')
del d

!cd /content && python -m pytest alphatrain/tests/test_model.py -v --tb=short 2>&1 | tail -5

In [ ]:
# Pillar 2Q: Pure self-play (v4+v5 combined), 15 epochs
# Warm start from 2P epoch 6
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train \
    --tensor-file alphatrain/data/selfplay.pt \
    --gpu-data --amp --compile \
    --value-bins 128 --value-channels 32 --value-hidden 512 --value-dropout 0.3 \
    --val-weight 0.01 --rank-weight 1.0 \
    --endgame-fraction 0.3 --endgame-threshold 100 \
    --adversarial-ranking \
    --resume alphatrain/data/model.pt --warm-start \
    --epochs 15 --batch-size 8192 --lr 1e-4 --warmup-epochs 2 \
    --copy-to /content/drive/MyDrive/alphatrain/pillar2q_best.pt \
    --save-dir /content/checkpoints/pillar2q

In [ ]:
# Copy ALL epoch checkpoints to Drive for bracket eval
import shutil, os, glob
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in sorted(glob.glob('/content/checkpoints/pillar2q/epoch_*.pt')):
    dst = f'{DRIVE}/pillar2q_{os.path.basename(f)}'
    shutil.copy(f, dst)
    print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')
# Also copy best and latest
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/pillar2q/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/pillar2q_{f}')